# PTCG 卡池分析

数据源：`EN_Card_Data.csv`（人类可读版卡表；引擎内 `all_card_data()/all_attack()` 是结构化版，二者 Card ID 对应）。

要点：**一行 = 一张卡的一个招式**（一张卡最多 3 行）；能量费用用 `●`(无色) + `{X}`(有色) 表示；Dragon 属性在 Type 列是日文「竜」。

In [1]:
import pandas as pd, re
from pathlib import Path
from collections import Counter

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

# 路径：兼容从 data/ 或 仓库根 运行
_cands = [Path.cwd() / 'EN_Card_Data.csv', Path.cwd() / 'data' / 'EN_Card_Data.csv']
CSV = next((p for p in _cands if p.exists()), _cands[-1])
ROOT = CSV.resolve().parent.parent
DECK = ROOT / 'sample_submission' / 'deck.csv'
print('CSV :', CSV)
print('DECK:', DECK)

STAGE = 'Stage (Pokémon)/Type (Energy and Trainer)'   # 进化阶段 / 卡类型 列

df = pd.read_csv(CSV)
print('原始表:', df.shape, '| 唯一卡数:', df['Card ID'].nunique())
df.head()

CSV : /Users/lucas/Documents/Projects/PTCG-AI-Battle-Challenge/data/EN_Card_Data.csv
DECK: /Users/lucas/Documents/Projects/PTCG-AI-Battle-Challenge/sample_submission/deck.csv
原始表: (2022, 17) | 唯一卡数: 1267


,Card ID,Card Name,Expansion,Collection No.,Stage (Pokémon)/Type (Energy and Trainer),Rule,Category,Previous stage,HP,Type,Weakness,Resistance (Type),Retreat,Move Name,Cost,Damage,Effect Explanation
0,1,Basic {G} Energy,SVE,1,Basic Energy,NaN,NaN,NaN,NaN,{G},NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Basic {R} Energy,SVE,2,Basic Energy,NaN,NaN,NaN,NaN,{R},NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Basic {W} Energy,SVE,3,Basic Energy,NaN,NaN,NaN,NaN,{W},NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Basic {L} Energy,SVE,4,Basic Energy,NaN,NaN,NaN,NaN,{L},NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Basic {P} Energy,SVE,5,Basic Energy,NaN,NaN,NaN,NaN,{P},NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# 卡级表：按 Card ID 去重（取每张卡第一行的卡面信息）
cards = df.drop_duplicates('Card ID').set_index('Card ID').copy()

# 大类：Pokemon / Trainer / Energy
def super_cat(s):
    s = str(s)
    if 'Pokémon' in s: return 'Pokemon'
    if s in ('Item', 'Supporter', 'Stadium', 'Pokémon Tool'): return 'Trainer'
    if 'Energy' in s: return 'Energy'
    return 'Other'
cards['super'] = cards[STAGE].map(super_cat)

# 能量属性规范化（Dragon 在 Type 列是「竜」）
def norm_type(t):
    if pd.isna(t): return None
    t = str(t).strip()
    return '{N}' if t == '竜' else t
cards['type_norm'] = cards['Type'].map(norm_type)

# 招式费用解析：返回总能量数；● = 无色，{X} = 有色
def parse_cost(c):
    if pd.isna(c): return 0
    colored = re.findall(r'\{[^}]*\}', str(c))
    colorless = str(c).count('●')
    return len(colored) + colorless

# 招式基础伤害：取前导数字（'30×' '60+' → 30/60；纯效果招式 → None）
def base_dmg(d):
    if pd.isna(d): return None
    m = re.match(r'\d+', str(d))
    return int(m.group()) if m else None

# 招式级表：只保留有招式名的行
moves = df[df['Move Name'].notna()].copy()
moves['energy_cost'] = moves['Cost'].map(parse_cost)
moves['base_damage'] = moves['Damage'].map(base_dmg)
print('cards:', cards.shape, '| moves:', moves.shape)

cards: (1267, 18) | moves: (1811, 19)


## 1. 卡池总览

In [ ]:
print('【大类】');          print(cards['super'].value_counts(), '\n')
print('【类型 / 进化阶段】'); print(cards[STAGE].value_counts(), '\n')
print('【特殊规则卡（Rule Box）】')
for r in ['Pokémon ex', 'Mega Pokémon ex', 'ACE SPEC']:
    print('  ', r, ':', (cards['Rule'] == r).sum())
print('\n【宝可梦属性分布】')
print(cards.loc[cards['super'] == 'Pokemon', 'type_norm'].value_counts())

【大类】
super
Pokemon    1083
Trainer     164
Energy       20
Name: count, dtype: int64 

【类型 / 进化阶段】
Stage (Pokémon)/Type (Energy and Trainer)
Basic Pokémon      595
Stage 1 Pokémon    345
Stage 2 Pokémon    116
Item                77
Supporter           61
Pokémon Tool        27
Stadium             26
Special Energy      12
Basic Energy         8
Name: count, dtype: int64 

【特殊规则卡（Rule Box）】
   Pokémon ex : 121
   Mega Pokémon ex : 30
   ACE SPEC : 29

【宝可梦属性分布】
type_norm
{G}    157
{P}    138
{W}    137
{F}    121
{D}    116
{C}    104
{R}    103
{L}     76
{M}     69
{N}     35
Name: count, dtype: int64


## 2. 宝可梦分析（HP / 阶段 / ex）

In [4]:
poke = cards[cards['super'] == 'Pokemon'].copy()
poke['HP'] = pd.to_numeric(poke['HP'], errors='coerce')
print('宝可梦:', len(poke),
      '| Basic:', (poke[STAGE] == 'Basic Pokémon').sum(),
      '| Stage1:', (poke[STAGE] == 'Stage 1 Pokémon').sum(),
      '| Stage2:', (poke[STAGE] == 'Stage 2 Pokémon').sum())

print('\n【HP 最高的 Basic（适合首发主攻 / 承伤）】')
display(poke[poke[STAGE] == 'Basic Pokémon']
        .sort_values('HP', ascending=False)
        [['Card Name', 'HP', 'type_norm', 'Rule', 'Retreat']].head(10))

print('【ex / Mega ex（高输出，但被击倒送对手 2 / 3 张奖赏）】')
display(poke[poke['Rule'].isin(['Pokémon ex', 'Mega Pokémon ex'])]
        .sort_values('HP', ascending=False)
        [['Card Name', 'HP', 'type_norm', 'Rule', STAGE]].head(12))

宝可梦: 1083 | Basic: 595 | Stage1: 345 | Stage2: 116

【HP 最高的 Basic（适合首发主攻 / 承伤）】


,Card Name,HP,type_norm,Rule,Retreat
Card ID,,,,,
1056,Mega Zygarde ex,310.0,{F},Mega Pokémon ex,2.0
756,Mega Kangaskhan ex,300.0,{C},Mega Pokémon ex,3.0
687,Mega Absol ex,280.0,{D},Mega Pokémon ex,2.0
431,Team Rocket's Mewtwo ex,280.0,{P},Pokémon ex,3.0
754,Mega Latias ex,280.0,{N},Mega Pokémon ex,1.0
781,Mega Heracross ex,280.0,{G},Mega Pokémon ex,2.0
1006,Mega Audino ex,270.0,{C},Mega Pokémon ex,1.0
695,Mega Mawile ex,270.0,{M},Mega Pokémon ex,2.0
766,Mega Diancie ex,270.0,{P},Mega Pokémon ex,1.0


【ex / Mega ex（高输出，但被击倒送对手 2 / 3 张奖赏）】


,Card Name,HP,type_norm,Rule,Stage (Pokémon)/Type (Energy and Trainer)
Card ID,,,,,
932,Mega Emboar ex,380.0,{R},Mega Pokémon ex,Stage 2 Pokémon
652,Mega Venusaur ex,380.0,{G},Mega Pokémon ex,Stage 2 Pokémon
939,Mega Feraligatr ex,370.0,{W},Mega Pokémon ex,Stage 2 Pokémon
904,Mega Dragonite ex,370.0,{N},Mega Pokémon ex,Stage 2 Pokémon
747,Mega Gardevoir ex,360.0,{P},Mega Pokémon ex,Stage 2 Pokémon
928,Mega Charizard Y ex,360.0,{R},Mega Pokémon ex,Stage 2 Pokémon
919,Mega Meganium ex,360.0,{G},Mega Pokémon ex,Stage 2 Pokémon
790,Mega Charizard X ex,360.0,{R},Mega Pokémon ex,Stage 2 Pokémon
723,Mega Abomasnow ex,350.0,{W},Mega Pokémon ex,Stage 1 Pokémon


## 3. 招式与能量效率

In [5]:
att = moves[(moves['base_damage'].notna()) & (moves['base_damage'] > 0)].copy()
att['dpe'] = att['base_damage'] / att['energy_cost'].clip(lower=1)   # 伤害 / 能量

print('【基础伤害最高的招式】')
display(att.sort_values('base_damage', ascending=False)
        [['Card ID', 'Card Name', 'Move Name', 'Cost', 'energy_cost', 'Damage', STAGE]].head(12))

print('【单能量 Basic 攻击手（rule-based 早期最实用：上场即能输出）】')
b1 = att[(att[STAGE] == 'Basic Pokémon') & (att['energy_cost'] <= 1)]
display(b1.sort_values('base_damage', ascending=False)
        [['Card Name', 'Move Name', 'Cost', 'Damage', 'HP']].head(12))

print('【能量效率最高（要求 ≥2 能量、基础伤害 ≥80）】')
display(att[(att['energy_cost'] >= 2) & (att['base_damage'] >= 80)]
        .sort_values('dpe', ascending=False)
        [['Card Name', 'Move Name', 'Cost', 'Damage', 'energy_cost', 'dpe']].head(12))

【基础伤害最高的招式】


,Card ID,Card Name,Move Name,Cost,energy_cost,Damage,Stage (Pokémon)/Type (Energy and Trainer)
1934,1180,Core Memory,Geobuster,{F}{F}{F}{F},4,350,Pokémon Tool
1546,904,Mega Dragonite ex,Ryuno Glide,{W}{L}{L},3,330,Stage 2 Pokémon
1585,932,Mega Emboar ex,Crimson Blast,{R}{R}●,3,320,Stage 2 Pokémon
555,302,Salamence ex,Dragon Impact,{R}{W}●●,4,300,Stage 2 Pokémon
383,210,Pikachu ex,Topaz Bolt,{G}{L}{M},3,300,Basic Pokémon
1304,754,Mega Latias ex,Illusory Impulse,{R}{P}●,3,300,Basic Pokémon
1158,662,Mega Camerupt ex,Volcanic Meteor,{R}●●●,4,280,Stage 1 Pokémon
442,239,Flareon ex,Carnelian,{R}{W}{L},3,280,Stage 1 Pokémon
447,241,Vaporeon ex,Aquamarine,{R}{W}{L},3,280,Stage 1 Pokémon
454,244,Jolteon ex,Dravite,{R}{W}{L},3,280,Stage 1 Pokémon


【单能量 Basic 攻击手（rule-based 早期最实用：上场即能输出）】


,Card Name,Move Name,Cost,Damage,HP
571,Hop’s Cramorant,Fickle Spitting,●,120,110.0
1057,Sawk,Rising Chop,{F},90,110.0
1181,Solrock,Cosmic Beam,{F},70,110.0
315,Fan Rotom,Assault Landing,●,70,70.0
1013,Basculin,Bared Fangs,{W},50,90.0
1303,Mega Latias ex,Strafe,●,40,280.0
1094,Druddigon,Shred,●,40,120.0
144,Miraidon,Peak Acceleration,●,40,110.0
1405,Paldean Tauros,Raging Charge,{F},40×,130.0
277,Lapras ex,Power Splash,{W},40×,220.0


【能量效率最高（要求 ≥2 能量、基础伤害 ≥80）】


,Card Name,Move Name,Cost,Damage,energy_cost,dpe
428,Slaking ex,Great Swing,●●,280,2,140.000000
1184,Mega Lucario ex,Mega Brave,{F}{F},270,2,135.000000
688,Cynthia's Garchomp ex,Draconic Buster,{F}{F},260,2,130.000000
1619,Luxray ex,Volt Strike,{L}{L},250,2,125.000000
1335,Mega Gengar ex,Void Gale,{D}{D},230,2,115.000000
1692,Haxorus,Dragon Pulse,{F}{M},230,2,115.000000
1546,Mega Dragonite ex,Ryuno Glide,{W}{L}{L},330,3,110.000000
1585,Mega Emboar ex,Crimson Blast,{R}{R}●,320,3,106.666667
213,Dragapult ex,Phantom Dive,{R}{P},200,2,100.000000
597,Blaziken ex,Smolder-sault,{R}●,200,2,100.000000


## 4. 训练家卡（牌组引擎：过牌 / 检索 / 干扰）

In [6]:
for kind in ['Supporter', 'Item', 'Stadium', 'Pokémon Tool']:
    sub = cards[cards[STAGE] == kind]
    print('=====', kind, '(', len(sub), '张 ) =====')
    display(sub[['Card Name', 'Rule', 'Effect Explanation']].head(10))

===== Supporter ( 61 张 ) =====


,Card Name,Rule,Effect Explanation
Card ID,,,
1181,Billy & O'Nare,NaN,"Draw 2 cards. Then, if you have 10 or more cards in your hand, draw 2 more c..."
1182,Boss’s Orders,NaN,Switch in 1 of your opponent’s Benched Pokémon to the Active Spot.
1183,Perrin,NaN,"Reveal up to 2 Pokémon in your hand and put them into your deck. If you do, ..."
1184,Lana’s Aid,NaN,Put up to 3 in any combination of Pokémon that don’t have a Rule Box and Bas...
1185,Explorer’s Guidance,NaN,Look at the top 6 cards of your deck and put 2 of them into your hand. Disca...
1186,Eri,NaN,"Your opponent reveals their hand, and you discard up to 2 Item cards you fin..."
1187,Morty’s Conviction,NaN,You can use this card only if you discard another card from your hand.\n\nDr...
1188,Ciphermaniac’s Codebreaking,NaN,"Search your deck for 2 cards, shuffle your deck, then put those cards on top..."
1189,Salvatore,NaN,Search your deck for a card that has no Abilities and evolves from 1 of your...


===== Item ( 77 张 ) =====


,Card Name,Rule,Effect Explanation
Card ID,,,
1077,Roto-Stick,NaN,Look at the top 4 cards of your deck. You may reveal any number of Supporter...
1078,Hole-Digging Shovel,NaN,Discard the top 2 cards of your deck.
1079,Rare Candy,NaN,Choose 1 of your Basic Pokémon in play. If you have a Stage 2 card in your h...
1080,Unfair Stamp,ACE SPEC,You can use this card only if any of your Pokémon were Knocked Out during yo...
1081,Enhanced Hammer,NaN,Discard a Special Energy from 1 of your opponent’s Pokémon.
1082,Hyper Aroma,ACE SPEC,"Search your deck for up to 3 Stage 1 Pokémon, reveal them, and put them into..."
1083,Love Ball,NaN,Search your deck for a Pokémon with the same name as 1 of your opponent’s Po...
1084,Boxed Order,NaN,"Search your deck for up to 2 Item cards, reveal them, and put them into your..."
1085,Awakening Drum,ACE SPEC,Draw a card for each of your Ancient Pokémon in play.


===== Stadium ( 26 张 ) =====


,Card Name,Rule,Effect Explanation
Card ID,,,
1242,Community Center,NaN,"Once during each player’s turn, if they played a Supporter card from their h..."
1243,Perilous Jungle,NaN,"During Pokémon Checkup, put 2 more damage counters on each Poisoned non-{D} ..."
1244,Full Metal Lab,NaN,{M} Pokémon (both yours and your opponent’s) take 30 less damage from attack...
1245,Festival Grounds,NaN,Each Pokémon that has any Energy attached (both yours and your opponent’s) r...
1246,Jamming Tower,NaN,Pokémon Tools attached to each Pokémon (both yours and your opponent’s) have...
1247,Neutralization Zone,ACE SPEC,Prevent all damage done to Pokémon that don’t have a Rule Box (both yours an...
1248,Academy at Night,NaN,"Once during each player’s turn, that player may put a card from their hand o..."
1249,Grand Tree,ACE SPEC,"Once during each player’s turn, that player may search their deck for a Stag..."
1250,Area Zero Underdepths,NaN,Each player who has any Tera Pokémon in play can have up to 8 Pokémon on the...


===== Pokémon Tool ( 27 张 ) =====


,Card Name,Rule,Effect Explanation
Card ID,,,
1154,Team Rocket’s Hypnotizer,NaN,If the Team Rocket’s Pokémon this card is attached to is in the Active Spot ...
1155,Survival Brace,ACE SPEC,If the Pokémon this card is attached to has full HP and would be Knocked Out...
1156,Lucky Helmet,NaN,If the Pokémon this card is attached to is in the Active Spot and is damaged...
1157,Rescue Board,NaN,The Retreat Cost of the Pokémon this card is attached to is {C} less. If tha...
1158,Maximum Belt,ACE SPEC,Attacks used by the Pokémon this card is attached to do 50 more damage to yo...
1159,Hero’s Cape,ACE SPEC,The Pokémon this card is attached to gets +100 HP.
1160,Heavy Baton,NaN,"If the Pokémon this card is attached to has a Retreat Cost of exactly 4, is ..."
1161,Handheld Fan,NaN,If the Pokémon this card is attached to is in the Active Spot and is damaged...
1162,Binding Mochi,NaN,Attacks used by the Poisoned Pokémon this card is attached to do 40 more dam...


## 5. 能量卡

In [7]:
print('【基础能量】')
display(cards[cards[STAGE] == 'Basic Energy'][['Card Name', 'type_norm']])
print('【特殊能量】')
display(cards[cards[STAGE] == 'Special Energy'][['Card Name', 'Effect Explanation']])

【基础能量】


,Card Name,type_norm
Card ID,,
1,Basic {G} Energy,{G}
2,Basic {R} Energy,{R}
3,Basic {W} Energy,{W}
4,Basic {L} Energy,{L}
5,Basic {P} Energy,{P}
6,Basic {F} Energy,{F}
7,Basic {D} Energy,{D}
8,Basic {M} Energy,{M}


【特殊能量】


,Card Name,Effect Explanation
Card ID,,
9,Boomerang Energy,"As long as this card is attached to a Pokémon, it provides {C} Energy.\n\nIf..."
10,Neo Upper Energy,"As long as this card is attached to a Pokémon, it provides {C} Energy.\n\nIf..."
11,Mist Energy,"As long as this card is attached to a Pokémon, it provides {C} Energy.\n\nPr..."
12,Legacy Energy,"As long as this card is attached to a Pokémon, it provides every type of Ene..."
13,Enriching Energy,"As long as this card is attached to a Pokémon, it provides {C} Energy.\n\nWh..."
14,Spiky Energy,"As long as this card is attached to a Pokémon, it provides {C} Energy.\n\nIf..."
15,Team Rocket's Energy,This card can only be attached to a Team Rocket’s Pokémon. If this card is a...
16,Prism Energy,"As long as this card is attached to a Pokémon, it provides {C} Energy.\n\nIf..."
17,Ignition Energy,"If this card is attached to 1 of your Pokémon, discard it at the end of your..."


## 6. 弱点 / 抗性分布

In [8]:
print('【弱点分布】（卡池里大量宝可梦弱这些属性 → 选这些属性当主攻很吃香）')
print(poke['Weakness'].value_counts().head(12))
print('\n【抗性分布】')
print(poke['Resistance (Type)'].value_counts().head(12))

【弱点分布】（卡池里大量宝可梦弱这些属性 → 选这些属性当主攻很吃香）
Weakness
{R}    220
{F}    188
{L}    155
{G}    141
{W}     99
{D}     91
{M}     86
{P}     41
Name: count, dtype: int64

【抗性分布】
Resistance (Type)
{F}    155
{G}     65
Name: count, dtype: int64


## 7. 现有 deck.csv 体检

In [9]:
deck = [int(x) for x in DECK.read_text().split() if x.strip()]
assert len(deck) == 60, '牌组不是 60 张: ' + str(len(deck))
comp = Counter(deck)

rows = []
for cid, n in comp.most_common():
    if cid in cards.index:
        r = cards.loc[cid]
        rows.append({'n': n, 'id': cid, 'name': r['Card Name'], 'kind': r[STAGE],
                     'rule': r['Rule'] if pd.notna(r['Rule']) else '',
                     'type': r['type_norm'], 'super': r['super']})
    else:
        rows.append({'n': n, 'id': cid, 'name': '<未找到>', 'kind': '', 'rule': '', 'type': '', 'super': 'Other'})
deck_df = pd.DataFrame(rows)
display(deck_df)

n_basic = sum(n for n, k in zip(deck_df['n'], deck_df['kind']) if k == 'Basic Pokémon')
print('【结构】按大类张数:')
print(deck_df.groupby('super')['n'].sum())
print('\n宝可梦:', deck_df.loc[deck_df['super'] == 'Pokemon', 'n'].sum(),
      '| 训练家:', deck_df.loc[deck_df['super'] == 'Trainer', 'n'].sum(),
      '| 能量:', deck_df.loc[deck_df['super'] == 'Energy', 'n'].sum())
print('可首发的 Basic 宝可梦张数:', n_basic, '（太少 → 起手无 Basic 的 mulligan 概率高）')

,n,id,name,kind,rule,type,super
0,35,3,Basic {W} Energy,Basic Energy,,{W},Energy
1,4,722,Snover,Basic Pokémon,,{W},Pokemon
2,4,723,Mega Abomasnow ex,Stage 1 Pokémon,Mega Pokémon ex,{W},Pokemon
3,4,1145,Mega Signal,Item,,NaN,Trainer
4,4,1227,Lillie's Determination,Supporter,,NaN,Trainer
5,4,1235,Waitress,Supporter,,NaN,Trainer
6,2,721,Kyogre,Basic Pokémon,,{W},Pokemon
7,2,1205,Cyrano,Supporter,,NaN,Trainer
8,1,1158,Maximum Belt,Pokémon Tool,ACE SPEC,NaN,Pokemon


【结构】按大类张数:
super
Energy     35
Pokemon    11
Trainer    14
Name: n, dtype: int64

宝可梦: 11 | 训练家: 14 | 能量: 35
可首发的 Basic 宝可梦张数: 6 （太少 → 起手无 Basic 的 mulligan 概率高）


## 8. 现有牌组的观察（待引擎实测验证）

当前 `deck.csv` 是一套 **Mega Abomasnow ex**（723，水/冰）牌：Snover(722)→Mega Abomasnow ex 进化线 + Kyogre(721) 副攻 + 10 张 Supporter + Mega Signal / Maximum Belt。明显的占位问题：

- **35 张水能量太多**（常规约 8–15 张）→ 挤占宝可梦 / 训练家位置，过牌质量差。
- **可首发 Basic 宝可梦只有 6 张**（4 Snover + 2 Kyogre）→ 起手无 Basic 的 **mulligan 概率偏高**，开局不稳。
- **进化线偏薄**：主攻是 Stage 1 的 Mega ex，依赖抽到 Snover 并进化，缺检索 / 铺场。

→ 后续组卡方向：压低能量数、增加 Basic 攻击手或检索（Supporter / Item）、提高一致性。**真正强弱要在远程 Linux 上跑自对战验证。**

## 9. 速查工具

In [10]:
# 按 Card ID 看卡面 + 全部招式
def card(cid):
    cols = ['Card ID', 'Card Name', STAGE, 'Rule', 'HP', 'Type',
            'Weakness', 'Retreat', 'Move Name', 'Cost', 'Damage', 'Effect Explanation']
    return df[df['Card ID'] == cid][cols]

# 按名字模糊查 Card ID
def find(name):
    hit = cards[cards['Card Name'].str.contains(name, case=False, na=False)]
    return hit[['Card Name', STAGE, 'Rule', 'HP', 'type_norm']]

# 示例：现有主攻 Mega Abomasnow ex
card(723)

,Card ID,Card Name,Stage (Pokémon)/Type (Energy and Trainer),Rule,HP,Type,Weakness,Retreat,Move Name,Cost,Damage,Effect Explanation
1255,723,Mega Abomasnow ex,Stage 1 Pokémon,Mega Pokémon ex,350.0,{W},{M},4.0,Hammer-lanche,{W}{W},100×,"Discard the top 6 cards of your deck, and this attack does 100 damage for ea..."
1256,723,Mega Abomasnow ex,Stage 1 Pokémon,Mega Pokémon ex,350.0,{W},{M},4.0,Frost Barrier,{W}{W}{W},200,"During your opponent’s next turn, this Pokémon takes 30 less damage from att..."
